In [1]:
import customtkinter as ctk
from tkinter import messagebox
from collections import deque
import random

# Thiết lập giao diện Dark Mode chuẩn công nghệ
ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")

GOAL = ((1, 2, 3), (8, 0, 4), (7, 6, 5))
MOVES = {"UP": (-1, 0), "DOWN": (1, 0), "LEFT": (0, -1), "RIGHT": (0, 1)}

# --- LOGIC THUẬT TOÁN ---
def find_blank(state):
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0: return i, j
    return None

def get_blankcanmoves(state):
    state_move = []
    x, y = find_blank(state)
    for move_name, (dx, dy) in MOVES.items():
        new_x, new_y = x + dx, y + dy
        if 0 <= new_x < 3 and 0 <= new_y < 3:
            matrix = [list(row) for row in state]
            matrix[x][y], matrix[new_x][new_y] = matrix[new_x][new_y], matrix[x][y]
            state_move.append(tuple(tuple(row) for row in matrix))
    return state_move

def get_move_direction(old_state, new_state):
    r1, c1 = find_blank(old_state)
    r2, c2 = find_blank(new_state)
    if r2 - r1 == 1: return "XUỐNG"
    if r1 - r2 == 1: return "LÊN"
    if c2 - c1 == 1: return "PHẢI"
    if c1 - c2 == 1: return "TRÁI"
    return "CHẠM"

def DLS(state, limit, stats):
    frontier = []
    frontier.append((state, [state]))
    reached = {state: 0}
    cutoff = False

    while frontier:
        current_state, path = frontier.pop()
        stats['visited'] += 1

        if current_state == GOAL: return path
        if len(path) - 1 == limit: cutoff = True
        else:
            for next_step in get_blankcanmoves(current_state):
                depth = len(path)
                if next_step not in reached or depth < reached[next_step]:
                    reached[next_step] = depth
                    frontier.append((next_step, path + [next_step]))
    return "CUTOFF" if cutoff else None

def IDS(state, stats):
    for limit in range(50):
        result = DLS(state, limit, stats)
        if result != "CUTOFF" and result is not None:
            return result
    return None

# --- HÀM HỖ TRỢ IN MA TRẬN ĐẸP MẮT ---
def format_board(matrix):
    res = ""
    for row in matrix:
        res += "   " + "  ".join(str(val) if val != 0 else "_" for val in row) + "\n"
    return res

# --- GIAO DIỆN APP DASHBOARD ---
class PuzzleApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title("AI 8-Puzzle Dashboard")
        self.geometry("760x570") # Tăng nhẹ chiều cao để chứa bộ đếm
        self.resizable(False, False)

        self.current_state = [list(row) for row in GOAL]
        self.grid_buttons = []
        self.player_steps = 0 # Khởi tạo biến đếm bước của người chơi

        self.grid_columnconfigure(0, weight=2)
        self.grid_columnconfigure(1, weight=1)

        self.create_widgets()
        self.shuffle_board()

    def create_widgets(self):
        # 1. BÊN TRÁI: KHU VỰC BÀN CỜ
        self.left_frame = ctk.CTkFrame(self, fg_color="transparent")
        self.left_frame.grid(row=0, column=0, sticky="nsew", padx=20, pady=20)

        ctk.CTkLabel(self.left_frame, text="8-PUZZLE IDS SOLVER", font=("Helvetica", 24, "bold")).pack(pady=10)

        self.board_frame = ctk.CTkFrame(self.left_frame, width=300, height=300, corner_radius=15)
        self.board_frame.pack(pady=10)

        for i in range(3):
            row_btns = []
            for j in range(3):
                btn = ctk.CTkButton(self.board_frame, text="", font=("Helvetica", 28, "bold"),
                                   width=85, height=85, corner_radius=10,
                                   command=lambda r=i, c=j: self.player_click(r, c))
                btn.grid(row=i, column=j, padx=6, pady=6)
                row_btns.append(btn)
            self.grid_buttons.append(row_btns)

        # Thêm nhãn hiển thị số bước người chơi tự đi
        self.player_step_label = ctk.CTkLabel(self.left_frame, text="Số bước bạn đi: 0", font=("Helvetica", 15, "bold"), text_color="#f39c12")
        self.player_step_label.pack(pady=(5, 0))

        self.status_text = ctk.CTkLabel(self.left_frame, text="Sẵn sàng!", font=("Helvetica", 14), text_color="#bdc3c7")
        self.status_text.pack(pady=(5, 10))

        btn_frame = ctk.CTkFrame(self.left_frame, fg_color="transparent")
        btn_frame.pack(pady=5)

        self.btn_shuffle = ctk.CTkButton(btn_frame, text="Xáo Trộn", command=self.shuffle_board, width=120, height=35, corner_radius=8)
        self.btn_shuffle.grid(row=0, column=0, padx=10)

        self.btn_solve = ctk.CTkButton(btn_frame, text="Giải IDS", fg_color="#27ae60", hover_color="#2ecc71",
                                       command=self.solve_ai, width=120, height=35, corner_radius=8)
        self.btn_solve.grid(row=0, column=1, padx=10)

        # 2. BÊN PHẢI: SIDEBAR THỐNG KÊ (DASHBOARD)
        self.right_frame = ctk.CTkFrame(self, width=280, corner_radius=15, fg_color="#242424", border_width=1, border_color="#333333")
        self.right_frame.grid(row=0, column=1, sticky="nsew", padx=(0, 20), pady=20)

        ctk.CTkLabel(self.right_frame, text="📊 AI DASHBOARD", font=("Helvetica", 18, "bold"), text_color="#3498db").pack(pady=(20, 15))

        # --- THẺ 1: SỐ NODE ĐÃ XÉT ---
        card1 = ctk.CTkFrame(self.right_frame, corner_radius=10, fg_color="#2b2b2b")
        card1.pack(fill="x", padx=15, pady=(0, 10))

        ctk.CTkLabel(card1, text="Trạng thái đã xét (Reached)", font=("Helvetica", 12), text_color="#aaaaaa").pack(pady=(10,0))
        self.reached_val = ctk.StringVar(value="0")
        ctk.CTkLabel(card1, textvariable=self.reached_val, font=("Helvetica", 32, "bold"), text_color="#f39c12").pack(pady=(0, 10))

        # --- THẺ 2: BƯỚC ĐI & TIẾN ĐỘ ---
        card2 = ctk.CTkFrame(self.right_frame, corner_radius=10, fg_color="#2b2b2b")
        card2.pack(fill="x", padx=15, pady=(0, 10))

        ctk.CTkLabel(card2, text="Tiến trình AI giải", font=("Helvetica", 12), text_color="#aaaaaa").pack(pady=(10,0))
        self.step_val = ctk.StringVar(value="0 / 0")
        ctk.CTkLabel(card2, textvariable=self.step_val, font=("Helvetica", 28, "bold"), text_color="#2ecc71").pack(pady=(0, 5))

        self.progress_bar = ctk.CTkProgressBar(card2, height=8, progress_color="#2ecc71", fg_color="#404040")
        self.progress_bar.set(0)
        self.progress_bar.pack(fill="x", padx=20, pady=(0, 15))

        # --- LOG BOX ---
        ctk.CTkLabel(self.right_frame, text="📝 Trạng thái bàn cờ", font=("Helvetica", 13, "bold")).pack(pady=(5,5))
        self.log_box = ctk.CTkTextbox(self.right_frame, font=("Consolas", 13), fg_color="#1e1e1e", corner_radius=10, border_spacing=5)
        self.log_box.pack(fill="both", expand=True, padx=15, pady=(0, 15))
        self.log_box.configure(state="disabled")

    def add_log(self, msg):
        self.log_box.configure(state="normal")
        self.log_box.insert("end", f"{msg}\n")
        self.log_box.see("end")
        self.log_box.configure(state="disabled")

    def shuffle_board(self):
        # Reset bước đếm của người chơi khi xáo trộn
        self.player_steps = 0
        self.player_step_label.configure(text="Số bước bạn đi: 0")

        self.log_box.configure(state="normal")
        self.log_box.delete("1.0", "end")
        self.log_box.configure(state="disabled")
        self.add_log("Hệ thống đang xáo trộn...\n" + "-"*23)

        self.reached_val.set("0")
        self.step_val.set("0 / 0")
        self.progress_bar.set(0)

        current = GOAL
        previous = None
        for _ in range(88):
            neighbors = get_blankcanmoves(current)
            valid_moves = [n for n in neighbors if n != previous]
            if not valid_moves: valid_moves = neighbors
            previous = current
            current = random.choice(valid_moves)

        self.current_state = [list(row) for row in current]
        self.update_buttons_display()
        self.status_text.configure(text="Trạng thái: Đã xáo trộn đề bài!")

        board_str = format_board(self.current_state)
        self.add_log(f"TRẠNG THÁI BẮT ĐẦU:\n{board_str}")

    def update_buttons_display(self):
        for i in range(3):
            for j in range(3):
                v = self.current_state[i][j]
                self.grid_buttons[i][j].configure(
                    text=str(v) if v != 0 else "",
                    fg_color="#1f538d" if v != 0 else "#1a252f",
                    hover_color="#2980b9" if v != 0 else "#1a252f"
                )

    def player_click(self, r, c):
        x, y = find_blank(tuple(tuple(row) for row in self.current_state))

        # Nếu click hợp lệ
        if abs(r-x) + abs(c-y) == 1:
            # Tăng biến đếm và cập nhật UI
            self.player_steps += 1
            self.player_step_label.configure(text=f"Số bước bạn đi: {self.player_steps}")

            old = tuple(tuple(row) for row in self.current_state)
            self.current_state[x][y], self.current_state[r][c] = self.current_state[r][c], self.current_state[x][y]

            move_dir = get_move_direction(old, tuple(tuple(row) for row in self.current_state))
            board_str = format_board(self.current_state)
            self.add_log(f"Bạn kéo: {move_dir}\n{board_str}")

            self.update_buttons_display()

            if tuple(tuple(row) for row in self.current_state) == GOAL:
                messagebox.showinfo("Chúc mừng", f"Bạn đã tự giải thành công trong {self.player_steps} bước!")

    def solve_ai(self):
        self.status_text.configure(text="Đang phân tích dữ liệu (IDS)...")
        self.btn_shuffle.configure(state="disabled")
        self.btn_solve.configure(state="disabled")
        self.update()

        stats = {'visited': 0}
        path = IDS(tuple(tuple(row) for row in self.current_state), stats)

        if path:
            self.reached_val.set(f"{stats['visited']:,}")
            self.add_log(f"-> Tìm thấy đường đi: {len(path)-1} bước\n" + "="*23)
            self.animate(path, 0, len(path)-1)
        else:
            messagebox.showerror("Lỗi", "Không giải được!")
            self.btn_shuffle.configure(state="normal")
            self.btn_solve.configure(state="normal")

    def animate(self, path, cur, total):
        if not path:
            self.status_text.configure(text="Hoàn thành xuất sắc!", text_color="#2ecc71")
            self.btn_shuffle.configure(state="normal")
            self.btn_solve.configure(state="normal")
            return

        state = path.pop(0)
        old = tuple(tuple(row) for row in self.current_state)
        move = get_move_direction(old, state)

        self.current_state = [list(row) for row in state]
        self.step_val.set(f"{cur} / {total}")

        if total > 0:
            self.progress_bar.set(cur / total)

        if move != "CHẠM":
            board_str = format_board(self.current_state)
            self.add_log(f"[Bước {cur:02d}]: {move}\n{board_str}")

        self.update_buttons_display()
        self.after(350, lambda: self.animate(path, cur+1, total))

if __name__ == "__main__":
    app = PuzzleApp()
    app.mainloop()

ModuleNotFoundError: No module named 'customtkinter'